# Agent4Science: кросс-частотное обобщение модели классификации СРГ

📘 Учебный ноутбук для студентов — полный цикл работы AI-агента от постановки задачи до статьи.

🔍 **Исследовательский вопрос:** Насколько хорошо модель классификации качества изображений Сибирского радиогелиографа (СРГ), обученная на частоте 3000 МГц, обобщается на данные 6000 МГц и 12200 МГц без дообучения?

📊 **Главный результат:** Модель **НЕ обобщается**. Согласованность: `67.1%` (6000 МГц) и `54.2%` (12200 МГц) при пороге 90%. Bad-изображения переносятся хорошо (80–92%), Ok — плохо (22–56%).

---

## Часть 0. Теория: AI-агенты

### Что такое AI-агент?

AI-агент — это программа, которая автономно выполняет цепочку действий для достижения цели. В отличие от обычного чат-бота (вопрос-ответ), агент:
- имеет **память** о предыдущих шагах
- использует **инструменты** (tools) для взаимодействия с внешним миром
- следует **плану** или **инструкции** (prompt/contract)
- может **размышлять** (reasoning) и **корректировать** поведение

### Архитектура агента

```
┌────────────────────────────────────────────────┐
│                   AI-агент                     │
│                                                │
│  ┌──────────┐     ┌──────────┐                 │
│  │ Память   │     │ Системный│                 │
│  │ контекст │◄─── │ промпт   │                 │
│  └──────────┘     │ AGENT.md │                 │
│       │           └──────────┘                 │
│       ▼                                        │
│  ┌──────────┐    ┌───────────────────┐         │
│  │ Скиллы   │───►│ Инструменты       │         │
│  │ (skills) │    │ (MCP / functions) │         │
│  └──────────┘    └───────────────────┘         │
│                       │                        │
└───────────────────────┼────────────────────────┘
                        ▼
              Внешний мир (файлы, API, БД)
```

### Ключевые понятия

#### 1. Системный промпт (системная инструкция)
Это «конституция» агента — файл, который задаёт:
- **роль** агента (кем он работает)
- **правила** (что можно, что нельзя)
- **структуру проекта** (какие файлы за что отвечают)
- **пошаговый план** (какие этапы и в каком порядке)

В нашем проекте — это `AGENT.md`.

#### 2. Память агента

| Тип | Что это | Пример в проекте |
|-----|---------|------------------|
| **Контекстное окно** | Текущий диалог: последние N токенов, которые видит LLM | История чата с opencode |
| **Краткосрочная память** | Промежуточные результаты текущей сессии | `diary.md` (append-only журнал действий) |
| **Долгосрочная память** | Структурированные знания, которые переживают сессии | `STATUS.md`, `LIT.md`, `HYPOTHESIS.md`, `TZ.md` |
| **Внешняя память** | Данные, которые агент читает, но не меняет | `logs/*.jsonl`, `literature/*.pdf` |

**Контекстное окно** — это «рабочий стол» агента. Чем больше окно, тем больше шагов агент может помнить без обращения к файлам. 
Ограничение: дороговизна (каждый токен стоит денег/времени).

**Краткосрочная vs долгосрочная память:**
- Краткосрочная — «что я сделал 5 минут назад» (`diary.md`)
- Долгосрочная — «как устроен проект» (`AGENT.md`, `MOTIVATION.md`, `HYPOTHESIS.md`, `LIT.md`, `TZ.md`, `STATUS.md`)

#### 3. Скиллы (Skills)
Скиллы — это **специализированные инструкции** для конкретных типов задач. Они загружаются поверх системного промпта, когда агент понимает, что задача относится к определённой категории.

В нашем проекте 6 скиллов (`.opencode/skills/`):
- `literature/SKILL.md` — как делать обзор литературы: поиск → скачивание PDF → анализ → review
- `latex/SKILL.md` — как писать научные статьи: преамбула, теоремы, таблицы, компиляция
- `python/SKILL.md` — как писать и запускать Python-код: venv, test, style
- `markdown/SKILL.md` — как оформлять документацию: GFM, таблицы, ссылки
- `debug-loop/SKILL.md` — как отлаживать: write→run→observe→fix
- `lean/SKILL.md` — формальная верификация теорем

#### 4. Инструменты (Tools / MCP)

Инструменты — это **функции**, которые агент может вызывать для взаимодействия с внешним миром. 

**MCP (Model Context Protocol)** — открытый протокол от Anthropic, который стандартизирует, как агенты общаются с инструментами. MCP-сервер — это программа, которая запускается отдельно и предоставляет набор инструментов через JSON-RPC.

Схема работы MCP:
```
┌──────────┐   JSON-RPC (stdio)   ┌────────────────┐
│  Агент   │◄────────────────────►│  MCP-сервер    │
│ opencode │                      │ experiment_mcp │
└──────────┘                      └───────┬────────┘
        │                                │
        │ читает инструкцию              │ читает logs/*.jsonl
        │ из opencode.json               │
        ▼                                ▼
   ┌──────────┐                   ┌──────────┐
   │ AGENT.md │                   │  Данные  │
   └──────────┘                   └──────────┘
```

**Инструменты в нашем проекте (`tools/experiment_mcp.py`):**

| Инструмент | Что делает | Тип |
|-----------|-----------|------|
| `explore_data()` | Показывает структуру логов: каналы, количество триплетов, баланс классов | Чтение |
| `explore_temporal()` | Распределение триплетов по месяцам (~80/месяц) | Чтение |
| `compute_metrics()` | Все метрики H1–H9: согласованность, bootstrap CI, χ², стратификация, t-test, Spearman, ANOVA, baseline | Анализ |
| `generate_plots()` | 6 графиков результатов | Визуализация |
| `get_h1_summary()` | Проверка самосогласованности (S_self, CI) | Анализ |

#### 5. Какие файлы проекта к чему относятся

| Категория | Файлы |
|-----------|-------|
| **Системный промпт** (инструкция агента) | `AGENT.md` |
| **Постановка задачи** (мотивация) | `statement/MOTIVATION.md` |
| **Долгосрочная память** | `statement/LIT.md`, `statement/HYPOTHESIS.md`, `statement/TZ.md`, `statement/STATUS.md` |
| **Краткосрочная память** | `statement/diary.md` |
| **Скиллы** | `.opencode/skills/literature/SKILL.md`, `.opencode/skills/latex/SKILL.md`, `.opencode/skills/python/SKILL.md`, `.opencode/skills/markdown/SKILL.md`, `.opencode/skills/debug-loop/SKILL.md`, `.opencode/skills/lean/SKILL.md` |
| **Инструменты (MCP)** | `tools/experiment_mcp.py`, `opencode.json` (конфиг подключения) |
| **Внешние данные** (read-only) | `logs/h1_*.jsonl` (100 записей), `logs/main_*.jsonl` (3000 записей), `literature/*.pdf` |
| **Результаты** | `experiment/plots/*.png`, `results/agreement_table.csv`, `notes/paper/ai4math_ysda2026_template/example_paper.tex` |

📦 **Зависимости проекта:**
- Python 3.10+, opencode 1.17.7+
- `requests`, `numpy`, `pandas`, `scipy`, `matplotlib`, `mcp`
- Tectonic / LaTeX (для компиляции статьи)

### 🗺️ Карта скиллов и инструментов по этапам

| Этап | Название | Скиллы | MCP-инструменты |
|------|----------|--------|------------------|
| 1 | Литературный обзор | `literature` | — |
| 2 | Математическая постановка | `markdown` | — |
| 3 | Гипотезы H1–H9 | `markdown` | — |
| 4 | Техническое задание | `markdown` | — |
| 5 | Анализ данных | `python`, `debug-loop` | `explore_data`, `explore_temporal`, `compute_metrics`, `generate_plots`, `get_h1_summary` |
| 6 | Интерпретация | `markdown` | — |
| 7 | Написание статьи | `latex` | — |
| 8 | Компиляция PDF | `latex` | — |

---

## Часть 1. Установка окружения

### 1.1. Установка Python и opencode

**Linux / macOS (рекомендуется через `uv` для ускорения):**
```bash
# 1. Установка Python и Git (если не установлены)
sudo apt install python3 python3-venv git curl  # Ubuntu/Debian
brew install python3 git                         # macOS

# 2. Установка opencode
curl -fsSL https://opencode.ai/install | bash

# 3. Установка uv (быстрый менеджер пакетов и venv)
curl -fsSL https://astral.sh/uv/install.sh | bash

# 4. Клонирование проекта
git clone https://github.com/EgorovYaroslav/agent4science.git
cd agent4science

# 5. Создание окружения и установка зависимостей
uv venv && source .venv/bin/activate
uv pip install -r requirements.txt
```

**Windows (opencode Desktop):**
1. Скачать opencode Desktop с https://opencode.ai/download
2. Установить Python 3.10+ с https://www.python.org/downloads/ — при установке отметить «Add Python to PATH»
3. Установить Git с https://git-scm.com/download/win
4. Открыть PowerShell:
```powershell
git clone https://github.com/EgorovYaroslav/agent4science.git
cd agent4science
python -m venv .venv
.venv\Scripts\activate
pip install -r requirements.txt
```
5. Запустить opencode Desktop → открыть папку `agent4science`

### 1.2. Проверка установки
```bash
python3 --version      # Python 3.10.x или выше
opencode --version     # opencode 1.17.x или выше
.venv/bin/python -c "import mcp; print('MCP OK')"
```

---

## Часть 2. Структура проекта

Проект организован по принципу единственной ответственности (SOLID):
```
agent4science/
├── AGENT.md                  ← контракт агента (системный промпт)
├── opencode.json             ← конфиг MCP-сервера
├── requirements.txt          ← Python-зависимости
│
├── statement/                ← постановка задачи (долгосрочная память)
│   ├── MOTIVATION.md         ← вопрос исследования
│   ├── LIT.md                ← литературный обзор
│   ├── HYPOTHESIS.md         ← гипотезы H1–H9
│   ├── TZ.md                 ← техническое задание
│   ├── diary.md              ← журнал работ (краткосрочная память)
│   └── STATUS.md             ← текущий статус
│
├── tools/                    ← MCP-сервер и инструменты
│   └── experiment_mcp.py
│
├── .opencode/skills/         ← скиллы агента (6 шт.)
│   ├── latex/SKILL.md
│   ├── literature/SKILL.md
│   ├── python/SKILL.md
│   ├── markdown/SKILL.md
│   ├── debug-loop/SKILL.md
│   └── lean/SKILL.md
│
├── logs/                     ← данные эксперимента (read-only!)
│   ├── h1_*.jsonl            ← 100 записей самосогласованности
│   └── main_*.jsonl          ← 3000 записей кросс-частотного теста
├── literature/               ← PDF-статьи (read-only!)
├── experiment/plots/         ← результаты-графики (6 шт.)
├── results/                  ← итоговые метрики
│   └── agreement_table.csv
├── notes/paper/              ← статья LaTeX
│   └── ai4math_ysda2026_template/
│       ├── example_paper.tex
│       ├── example_paper.bib
│       ├── example_paper.pdf
│       └── figures/
└── materials/                ← примеры кода и клиенты API/FTP
```

---

## Часть 3. Этап 1: работа с литературой

**Вход:** `literature/2507.04211v1.pdf`  
**Выход:** `statement/LIT.md`

### 🛠️ Скиллы и инструменты этапа

| Тип | Название | Что делает |
|-----|----------|-----------|
| 📘 **Скилл** | `literature/SKILL.md` | Инструктирует агента: как читать PDF, извлекать тезисы, искать gap analysis, оформлять IEEE-цитаты |
| 🔧 **Инструменты** | *(встроенные opencode)* | Чтение PDF (`pdftotext`), запись в `statement/LIT.md` |
| 🌐 **MCP** | — | Не используется (нет внешних данных) |

**Что делает агент:**
1. Читает PDF (извлекает текст через `pdftotext`)
2. Извлекает ключевые тезисы: архитектура модели, метрики, датасет
3. Ищет `gap analysis`: чего НЕ сделано в этой работе
4. Пишет `LIT.md` с IEEE-цитатами

**Важно для студентов:**
- Агент НЕ выдумывает содержание — он читает реальный PDF
- Если PDF сканированный (без OCR), агент не может его прочитать
- `LIT.md` — это долгосрочная память: он останется после завершения сессии

**Запуск:** в чате opencode написать: `выполни этап 1`

---

## Часть 4. Этап 2: математическая постановка

**Вход:** `statement/MOTIVATION.md`, `statement/LIT.md`  
**Выход:** заметки в `statement/diary.md` (блоки M.0–M.9)

### 🛠️ Скиллы и инструменты этапа

| Тип | Название | Что делает |
|-----|----------|-----------|
| 📘 **Скилл** | `markdown/SKILL.md` | Правила оформления Markdown: заголовки, списки, таблицы, математика в `$...$` |
| 🔧 **Инструменты** | *(встроенные opencode)* | Чтение `MOTIVATION.md`/`LIT.md`, запись в `diary.md` |
| 🌐 **MCP** | — | Не используется |

**Что делает агент:**
1. Выписывает все обозначения (M.0): частоты, переменные, метрики
2. Формулирует предположения (M.1): Δt ≤ 3 мин, seed=42, 1000 триплетов
3. Формулирует вопросы (M.2)
4. Описывает статистические методы (M.3)
5. Описывает ограничения (M.4)

**Учебный комментарий:**
На этом этапе агент НЕ принимает научных решений — он только структурирует то, что уже написано в `MOTIVATION.md`. 
Это иллюстрация принципа: **агент — исполнитель, а не исследователь**.

---

## Часть 5. Этап 3: гипотезы H1–H9

**Вход:** математическая постановка из `diary.md`  
**Выход:** `statement/HYPOTHESIS.md`

### 🛠️ Скиллы и инструменты этапа

| Тип | Название | Что делает |
|-----|----------|-----------|
| 📘 **Скилл** | `markdown/SKILL.md` | Оформление таблиц гипотез, формулировок, методов проверки |
| 🔧 **Инструменты** | *(встроенные opencode)* | Чтение `diary.md`, запись в `HYPOTHESIS.md` |
| 🌐 **MCP** | — | Не используется |

**Что делает агент:**
1. Формулирует 9 проверяемых гипотез
2. Для каждой указывает: формулировку, метод проверки, ожидаемый результат

| # | Суть | Ожидание |
|---|------|----------|
| H1 | Модель детерминирована? | S ≥ 0.95 |
| H2 | Симметрия согласованности | p ≥ 0.05 |
| H3 | Практические пороги | S ≥ 0.90 / 0.85 |
| H4 | Асимметрия по классам | Есть разница |
| H5 | Уверенность падает с частотой | p < 0.05 |
| H6 | Корреляция с Δt | ρ ≠ 0 |
| H7 | Распределения однородны | p ≥ 0.05 |
| H8 | Сезонная стабильность | p ≥ 0.05 |
| H9 | Модель > baseline | p < 0.05 |

---

## Часть 6. Этап 4: техническое задание

**Вход:** `statement/HYPOTHESIS.md`  
**Выход:** `statement/TZ.md`

### 🛠️ Скиллы и инструменты этапа

| Тип | Название | Что делает |
|-----|----------|-----------|
| 📘 **Скилл** | `markdown/SKILL.md` | Структурирование ТЗ: разделы, таблицы, списки |
| 🔧 **Инструменты** | *(встроенные opencode)* | Чтение `HYPOTHESIS.md`, запись в `TZ.md` |
| 🌐 **MCP** | — | Не используется |

**Что делает агент:**
1. Описывает источники данных (`logs/*.jsonl`)
2. Описывает MCP-инструменты для анализа
3. Составляет план эксперимента
4. Перечисляет deliverables

**Учебный комментарий:**
`TZ.md` — это «мост» между постановкой задачи и выполнением. 
В классическом ML-проекте аналог — pipeline specification.

---

## Часть 7. Этап 5: анализ данных через MCP

**Вход:** `statement/TZ.md`  
**Выход:** 6 графиков в `experiment/plots/`, метрики в `statement/diary.md`

### 🛠️ Скиллы и инструменты этапа

| Тип | Название | Что делает |
|-----|----------|-----------|
| 📘 **Скилл** | `python/SKILL.md` | Правила работы с Python: venv, style, обработка ошибок |
| 📘 **Скилл** | `debug-loop/SKILL.md` | Цикл отладки: write → run → observe → fix (если MCP-вызов упал) |
| 🌐 **MCP** | `explore_data()` | Чтение: структура логов, каналы, баланс классов |
| 🌐 **MCP** | `explore_temporal()` | Чтение: распределение триплетов по месяцам |
| 🌐 **MCP** | `compute_metrics()` | Анализ: все метрики H1–H9, bootstrap CI, χ², t-test, Spearman, ANOVA |
| 🌐 **MCP** | `generate_plots()` | Визуализация: 6 графиков → `experiment/plots/*.png` |
| 🌐 **MCP** | `get_h1_summary()` | Анализ: самосогласованность (S_self, CI) |

⚠️ **Это единственный этап, где агент реально вызывает MCP-инструменты.** На всех остальных этапах он работает только с файлами.

### 📊 Результаты эксперимента

| Гипотеза | Формулировка | Результат | Статистика |
|----------|--------------|-----------|------------|
| **H1** | Самосогласованность ≥ 0.95 | ✅ `S = 1.0000` | CI: [1.000, 1.000] |
| **H2** | S(3000,6000) = S(3000,12200) | ❌ не равны | χ² = 34.9, p = 3.5e-9 |
| **H3** | S ≥ 0.90 (6000) и ≥ 0.85 (12200) | ❌ пороги не пройдены | z_6 = −24.1, z_12 = −27.3 |
| **H4** | Асимметрия Ok vs Bad | ✅ Bad: 80–92%, Ok: 22–56% | Сильная асимметрия |
| **H5** | Уверенность падает с частотой | ✅ значимо | t = 6.93, p = 7.4e-12 |
| **H6** | Корреляция с Δt < 0 | ❌ нет корреляции | ρ ≈ 0.02–0.06 |
| **H7** | Распределения меток однородны | ❌ сдвиг | Ok rate: 54%→40%→16%, p ≈ 0 |
| **H8** | Сезонная стабильность | ❌ нестабильно | 37–100%, p = 3.6e-28 |
| **H9** | Модель > majority baseline | ✅ модель лучше | p ≈ 0 для обоих каналов |

---

## Часть 8. Этап 6: интерпретация

**Вход:** графики, логи, метрики  
**Выход:** заметки для автора в `statement/diary.md` (M.5)

### 🛠️ Скиллы и инструменты этапа

| Тип | Название | Что делает |
|-----|----------|-----------|
| 📘 **Скилл** | `markdown/SKILL.md` | Оформление раздела интерпретации в `diary.md` |
| 🔧 **Инструменты** | *(встроенные opencode)* | Чтение графиков/метрик, запись в `diary.md` |
| 🌐 **MCP** | — | Не используется (данные уже собраны) |

⚠️ **Важнейший принцип:**
> Интерпретация — работа автора, не агента! 
> Агент предоставляет данные и факты, а вопрос «что значит этот результат?» решает исследователь.

**Что делает агент:**
- Собирает все числа в один блок
- Формулирует, какие гипотезы подтвердились, какие нет
- Предлагает возможные физические объяснения (со ссылками на литературу)

---

## Часть 9. Этап 7: написание статьи

**Вход:** `statement/` (все md-файлы) + `experiment/plots/`  
**Выход:** `notes/paper/ai4math_ysda2026_template/example_paper.tex`

### 🛠️ Скиллы и инструменты этапа

| Тип | Название | Что делает |
|-----|----------|-----------|
| 📘 **Скилл** | `latex/SKILL.md` | Правила LaTeX: преамбула, секции, таблицы, теоремы, библиография, подключение графиков через `\includegraphics` |
| 🔧 **Инструменты** | *(встроенные opencode)* | Чтение всех md-файлов и PNG-графиков, запись в `.tex` и `.bib` |
| 🌐 **MCP** | — | Не используется |

**Что делает агент:**
1. Берёт математическую постановку из `statement/`
2. Добавляет результаты экспериментов (графики, цифры)
3. Заполняет `example_paper.tex` по структуре шаблона конференции
4. Обновляет `example_paper.bib`

**Структура статьи:**
- Abstract (автоматически сгенерирован из результатов)
- Introduction (постановка задачи)
- Related Work (из LIT.md)
- Methodology (из TZ.md и HYPOTHESIS.md)
- Results (числа из `compute_metrics` + 6 графиков)
- Discussion (агент пишет факты, автор — анализ)
- Conclusion

---

## Часть 10. Этап 8: компиляция PDF

**Вход:** `notes/paper/ai4math_ysda2026_template/example_paper.tex`  
**Выход:** `example_paper.pdf`

### 🛠️ Скиллы и инструменты этапа

| Тип | Название | Что делает |
|-----|----------|-----------|
| 📘 **Скилл** | `latex/SKILL.md` | Инструкции по компиляции: выбор движка, обработка ошибок, пути к `figures/` |
| 🔧 **Инструменты** | `tectonic` или `pdflatex` | Внешние CLI-инструменты (вызываются через shell opencode) |
| 🌐 **MCP** | — | Не используется |

### Вариант A: Tectonic (Linux / macOS) — самый простой
Не требует установки полного LaTeX-дистрибутива. Работает «из коробки»:
```bash
curl -fsSL https://github.com/tectonic-typesetting/tectonic/releases/download/tectonic%400.15.0/tectonic-0.15.0-x86_64-unknown-linux-gnu.tar.gz | tar -xz
./tectonic -X compile notes/paper/ai4math_ysda2026_template/example_paper.tex
```

### Вариант B: pdflatex (Linux / macOS)
```bash
sudo apt install texlive-latex-recommended texlive-fonts-recommended
cd notes/paper/ai4math_ysda2026_template
pdflatex -interaction=nonstopmode -halt-on-error example_paper.tex
```

### Вариант C: Windows (рекомендуется Overleaf)
1. Зайти на https://www.overleaf.com
2. New Project → Upload Project → выбрать папку `notes/paper/ai4math_ysda2026_template/`
3. Нажать «Recompile» (автоматически подхватит `ai4math_ysda2026.sty` и библиографию)
4. Графики из `figures/` подключатся автоматически.

---

## Часть 11. Адаптация под своё исследование

1. Запустить `python3 clear.py` — удалит все результаты, оставит структуру.
2. Отредактировать ключевые файлы:

#### 📝 `AGENT.md` (контракт агента)
| Что менять | Где в файле |
|-----------|-------------|
| Название проекта | строка 4 — заголовок в описании роли |
| Входные данные | секция 7 — заменить описание СРГ, частот, API/FTP на свои |
| Исследовательские вопросы | секция 7 — перечислить свои вопросы |
| Гипотезы | секция 7 — таблица H1–H9 под вашу задачу |
| Структура проекта | секция 2 — если добавляете/убираете папки |
| Критерии готовности | секция 9 — убрать лишнее, добавить своё |

#### 🔬 `statement/MOTIVATION.md` (исследовательский вопрос)
| Что менять | Где в файле |
|-----------|-------------|
| Формулировка задачи | строка 5 — основной вопрос исследования |
| Формальное определение | строки 7–13 — переменные, датасеты, условия |
| Вопросы для ответа | строки 17–22 — перечислить свои |
| Источники данных | строки 47–51 — ссылки на API, FTP, БД |
| Ограничения | строки 44–56 — критерии включения, seed, объём выборки |
| Критерии успеха | строки 67–76 — что считается выполненной работой |

#### ⚙️ Другие файлы (опционально)
| Файл | Что делать |
|------|-----------|
| `opencode.json` | Поменять путь к MCP-серверу, если ваш скрипт называется иначе |
| `literature/` | Положить свои PDF, очистить папку |
| `logs/` | Заменить на свои JSONL-логи (формат: одна запись = один инференс) |
| `tools/experiment_mcp.py` | Переписать под свою логику чтения данных и метрики |
| `notes/paper/*/example_paper.tex` | Заменить шаблон конференции или убрать |

3. Запустить `opencode` и написать в чат: `Выполни все этапы из AGENT.md`

---

## Контрольные вопросы для студентов

1. **Чем AI-агент отличается от обычного чат-бота?**  
   *Ответ:* агент автономно выполняет цепочку действий, использует инструменты, имеет память и следует плану.

2. **Какие файлы проекта являются долгосрочной памятью агента?**  
   *Ответ:* `LIT.md`, `HYPOTHESIS.md`, `TZ.md`, `STATUS.md`, `MOTIVATION.md`, `AGENT.md`.

3. **Что такое MCP и зачем он нужен?**  
   *Ответ:* Model Context Protocol — стандарт для подключения инструментов к AI-агентам. MCP-сервер предоставляет функции (например, чтение логов, вычисление метрик), которые агент вызывает по мере необходимости.

4. **Почему агент не должен интерпретировать результаты?**  
   *Ответ:* интерпретация — научная работа, требующая экспертизы. Агент — лаборант/инженер, а не исследователь.

5. **Что такое контекстное окно и в чём его ограничение?**  
   *Ответ:* это количество токенов, которые LLM «видит» в текущем диалоге. Ограничение: чем больше окно, тем дороже и медленнее каждый запрос.

6. **Какую архитектуру имеет модель классификации СРГ?**  
   *Ответ:* ансамбль CLIP + EfficientNet + CatBoost + FFNN — итоговая точность ~95% на обучающей частоте.

7. **Почему модель плохо обобщается на 12200 МГц?**  
   *Ответ:* сильный сдвиг распределения — на 12200 МГц модель предсказывает `Bad` в 84% случаев (vs 46% на 3000 МГц). Разные частоты имеют разную физику излучения и условия наблюдения.

8. **На каком этапе агент впервые вызывает MCP-инструменты?**  
   *Ответ:* на этапе 5 (анализ данных). До этого он работал только с файлами через встроенные инструменты opencode.

9. **Какой скилл подключается на этапе написания статьи?**  
   *Ответ:* `latex/SKILL.md` — он задаёт правила оформления LaTeX-документа, преамбулы, таблиц, библиографии.

10. **Зачем нужен скилл `debug-loop/SKILL.md`?**  
    *Ответ:* он инструктирует агента, как отлаживать код: записать → запустить → посмотреть ошибку → исправить. Активен на этапе 5, если MCP-вызов упал.

---

👤 **Автор:** Ярослав Егоров, ИСЗФ СО РАН, Иркутск  
📜 **Лицензия:** MIT